In [1]:
!pip install -q streamlit transformers accelerate bitsandbytes huggingface_hub pyngrok
!npm install localtunnel

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 9.1/9.1 MB 62.9 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 60.7/60.7 MB 32.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 6.9/6.9 MB 125.2 MB/s eta 0:00:00
⠙⠹⠸⠼⠴⠦⠧⠇⠏⠋⠙⠹⠸⠼⠴
added 22 packages in 2s
⠴
⠴3 packages are looking for funding
⠴  run `npm fund` for details
⠴npm notice
npm notice New major version of npm available! 10.8.2 -> 11.10.1
npm notice Changelog: https://github.com/npm/cli/releases/tag/v11.10.1
npm notice To update run: npm install -g npm@11.10.1
npm notice
⠴

In [2]:
%%writefile app_final.py

import streamlit as st
import torch
import gc
from transformers import AutoTokenizer, AutoModelForCausalLM
from huggingface_hub import login
from dataclasses import dataclass, field
from typing import Dict, List, Optional, Any, Tuple
from enum import Enum
from datetime import datetime
import json
import uuid
import os
import re

# =============================================================================
# PAGE CONFIG
# =============================================================================
st.set_page_config(
    page_title="Med0wl - Patient Communication Engine",
    layout="wide",
    initial_sidebar_state="collapsed"
)

# =============================================================================
# CSS - Clean, Professional, No Gradients, No Emojis
# =============================================================================
st.markdown("""
<style>
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;500;600;700&display=swap');

* { font-family: 'Inter', sans-serif; }
#MainMenu, footer, header { visibility: hidden; }

/* Main Container */
.main-container { max-width: 1200px; margin: 0 auto; padding: 0 20px; }

/* Top Navigation */
.top-nav {
    background: #ffffff;
    border-bottom: 1px solid #e5e7eb;
    padding: 0;
    margin: -1rem -1rem 2rem -1rem;
    position: sticky;
    top: 0;
    z-index: 100;
}
.nav-inner {
    display: flex;
    justify-content: center;
    max-width: 800px;
    margin: 0 auto;
}

/* Category Pills */
.pills-container {
    display: flex;
    flex-wrap: wrap;
    gap: 8px;
    margin-bottom: 24px;
}

/* Section Header */
.section-title {
    display: flex;
    align-items: center;
    gap: 8px;
    padding: 16px 0;
    margin: 24px 0 16px 0;
    border-bottom: 1px solid #e5e7eb;
    font-size: 12px;
    font-weight: 600;
    color: #6b7280;
    text-transform: uppercase;
    letter-spacing: 0.5px;
}

/* Profile Cards */
.profile-grid { display: grid; grid-template-columns: repeat(2, 1fr); gap: 16px; }
.profile-card {
    background: #ffffff;
    border: 1px solid #e5e7eb;
    border-radius: 12px;
    padding: 20px;
    cursor: pointer;
    transition: all 0.2s;
}
.profile-card:hover { border-color: #2563eb; box-shadow: 0 4px 12px rgba(37,99,235,0.08); }
.profile-card.selected { border-color: #2563eb; background: #eff6ff; }
.profile-name { font-size: 16px; font-weight: 600; color: #111827; margin-bottom: 6px; }
.profile-desc { font-size: 13px; color: #6b7280; margin-bottom: 12px; line-height: 1.5; }
.profile-tags { display: flex; gap: 8px; flex-wrap: wrap; }
.tag {
    padding: 4px 10px;
    border-radius: 4px;
    font-size: 12px;
    font-weight: 500;
}
.tag-yellow { background: #fef3c7; color: #92400e; }
.tag-blue { background: #dbeafe; color: #1e40af; }
.tag-gray { background: #f3f4f6; color: #4b5563; }
.tag-pink { background: #fce7f3; color: #be185d; }
.tag-green { background: #d1fae5; color: #065f46; }
.tag-red { background: #fee2e2; color: #991b1b; }

/* Config Cards */
.config-grid { display: grid; grid-template-columns: repeat(2, 1fr); gap: 16px; margin-bottom: 16px; }

/* Slider Labels */
.slider-labels {
    display: flex;
    justify-content: space-between;
    margin-top: 8px;
    font-size: 12px;
    color: #6b7280;
}
.slider-value {
    text-align: center;
    color: #2563eb;
    font-weight: 600;
    font-size: 14px;
}

/* Results */
.result-card {
    background: transparent; /* Let the Streamlit theme handle the background */
    border: 1px solid #e5e7eb;
    border-radius: 12px;
    padding: 24px;
    margin-bottom: 16px;
}
.status-pending { background: #fef3c7; color: #92400e; padding: 6px 12px; border-radius: 20px; font-size: 12px; font-weight: 600; }
.status-approved { background: #d1fae5; color: #065f46; padding: 6px 12px; border-radius: 20px; font-size: 12px; font-weight: 600; }
.status-failed { background: #fee2e2; color: #991b1b; padding: 6px 12px; border-radius: 20px; font-size: 12px; font-weight: 600; }

.metrics { display: grid; grid-template-columns: repeat(4, 1fr); gap: 16px; }
.metric { text-align: center; padding: 16px; background: #f9fafb; border-radius: 8px; }
.metric-val { font-size: 24px; font-weight: 700; color: #111827; }
.metric-lbl { font-size: 12px; color: #6b7280; margin-top: 4px; }

.brochure-content {
    background: rgba(128, 128, 128, 0.05); /* Very light grey that works in both modes */
    border: 1px solid #e5e7eb;
    border-radius: 8px;
    padding: 24px;
    line-height: 1.7;
    white-space: pre-wrap;
    color: inherit; /* Forces the text to follow the user's Dark/Light theme */
}

/* Architecture Diagram */
.arch-step {
    background: #f9fafb;
    border: 1px solid #e5e7eb;
    border-radius: 8px;
    padding: 16px;
    margin-bottom: 12px;
}
.arch-step-title { font-weight: 600; color: #111827; font-size: 14px; }
.arch-step-desc { font-size: 13px; color: #6b7280; margin-top: 4px; }
.arch-arrow { text-align: center; color: #9ca3af; font-size: 20px; margin: 8px 0; }

/* Contraindication Alert */
.contra-alert {
    background: #fee2e2;
    border: 1px solid #fecaca;
    border-radius: 8px;
    padding: 16px;
    margin-bottom: 16px;
}
.contra-title { font-weight: 600; color: #991b1b; font-size: 14px; }
.contra-list { font-size: 13px; color: #7f1d1d; margin-top: 8px; }
</style>
""", unsafe_allow_html=True)

# =============================================================================
# CONFIG & CONSTANTS
# =============================================================================
try:
    from kaggle_secrets import UserSecretsClient
    HF_API_TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except:
    HF_API_TOKEN = os.getenv('HF_TOKEN', '')

# Medical contraindications database for FHIR safety checking
KNOWN_CONTRAINDICATIONS = {
    "drug_drug": [
        ("warfarin", "aspirin", "Increased bleeding risk"),
        ("warfarin", "ibuprofen", "Increased bleeding risk"),
        ("metformin", "contrast dye", "Lactic acidosis risk"),
        ("lisinopril", "spironolactone", "Hyperkalemia risk"),
        ("simvastatin", "clarithromycin", "Muscle toxicity risk"),
        ("amiodarone", "warfarin", "Enhanced anticoagulation"),
    ],
    "drug_allergy": [
        ("penicillin", "rash,anaphylaxis", "Severe allergic reaction"),
        ("sulfa", "rash,steven-johnson", "Severe skin reaction"),
        ("nsaid", "asthma,angioedema", "Bronchospasm risk"),
    ],
    "drug_condition": [
        ("beta-blocker", "asthma", "Bronchospasm risk"),
        ("nsaid", "kidney disease", "Nephrotoxicity risk"),
        ("metformin", "kidney disease", "Lactic acidosis"),
        ("anticholinergic", "glaucoma", "Acute angle-closure"),
    ]
}

# Attention markers for archetype-specific content adaptation
ATTENTION_MARKERS = {
    "pediatric_cancer_child": {
        "priority_keywords": ["gentle", "brave", "story", "color", "picture", "soft", "hug", "cuddle"],
        "avoid_keywords": ["survival rate", "mortality", "aggressive", "invasive", "terminal"],
        "emotional_triggers": ["fear", "pain", "alone", "scary"]
    },
    "pediatric_cancer_parent": {
        "priority_keywords": ["prognosis", "treatment options", "support group", "research", "outcomes"],
        "avoid_keywords": ["guaranteed cure", "simple", "easy"],
        "emotional_triggers": ["blame", "fault", "preventable"]
    },
    "depression": {
        "priority_keywords": ["hope", "support", "gradual improvement", "you are not alone"],
        "avoid_keywords": ["just cheer up", "think positive", "snap out of"],
        "emotional_triggers": ["weakness", "failure", "burden"]
    },
    "anxiety": {
        "priority_keywords": ["coping strategies", "grounding techniques", "breathing", "manageable"],
        "avoid_keywords": ["just relax", "calm down", "irrational"],
        "emotional_triggers": ["crazy", "overreacting", "dramatic"]
    },
    "autism": {
        "priority_keywords": ["routine", "sensory-friendly", "clear steps", "visual supports"],
        "avoid_keywords": ["cure", "fix", "normal", "recover"],
        "emotional_triggers": ["deficient", "disordered", "broken"]
    },
    "adhd": {
        "priority_keywords": ["strengths", "energy", "creative", "strategies", "tools"],
        "avoid_keywords": ["lazy", "unmotivated", "just focus"],
        "emotional_triggers": ["stupid", "careless", "disruptive"]
    },
    "dyslexia": {
        "priority_keywords": ["alternative formats", "audio", "visual", "strengths"],
        "avoid_keywords": ["slow reader", "not trying", "careless"],
        "emotional_triggers": ["dumb", "stupid", "lazy"]
    },
    "diabetes": {
        "priority_keywords": ["manageable", "monitoring", "lifestyle", "control"],
        "avoid_keywords": ["forbidden foods", "can never eat", "ruined life"],
        "emotional_triggers": ["your fault", "caused by eating", "deserved"]
    },
    "default": {
        "priority_keywords": ["clear", "understandable", "support", "information"],
        "avoid_keywords": ["guaranteed", "always", "never", "100%"],
        "emotional_triggers": ["hopeless", "pointless", "worthless"]
    }
}

# =============================================================================
# MODEL LOADING WITH CACHING
# =============================================================================
@st.cache_resource(show_spinner="Loading MedGemma into GPU...")
def load_medgemma(token):
    """Load MedGemma model with GPU optimization for Kaggle T4."""
    if token:
        login(token=token)
    tokenizer = AutoTokenizer.from_pretrained("google/medgemma-1.5-4b-it")
    model = AutoModelForCausalLM.from_pretrained(
        "google/medgemma-1.5-4b-it",
        device_map={"": 0},
        torch_dtype=torch.bfloat16,
        low_cpu_mem_usage=True # Add this to prevent CPU bottlenecks
    )
    return tokenizer, model

# =============================================================================
# DATA CLASSES
# =============================================================================
@dataclass
class PatientProfile:
    id: str
    name: str
    category: str
    description: str
    is_pediatric: bool = False
    is_dual_brochure: bool = False
    attention_markers: Dict = field(default_factory=dict)

@dataclass
class GeneratedBrochure:
    patient_content: str
    parent_content: Optional[str] = None
    verification_result: Dict = field(default_factory=dict)
    attention_metrics: Dict = field(default_factory=dict)
    session_id: str = field(default_factory=lambda: str(uuid.uuid4())[:8])
    profile_id: str = ""
    doctor_approved: bool = False
    generation_log: List[Dict] = field(default_factory=list)
    contraindications: List[Dict] = field(default_factory=list)

@dataclass
class Contraindication:
    type: str  # drug_drug, drug_allergy, drug_condition
    severity: str  # high, medium, low
    description: str
    involved_items: List[str]

# =============================================================================
# PROFILES - 28 Archetypes with Attention Markers
# =============================================================================
PROFILES = {
    "Pediatric": [
        PatientProfile("pediatric_cancer_child", "Pediatric Cancer - Child", "Pediatric", 
                      "Young cancer patient receiving treatment - age-appropriate, gentle communication",
                      is_pediatric=True, is_dual_brochure=True,
                      attention_markers=ATTENTION_MARKERS["pediatric_cancer_child"]),
        PatientProfile("pediatric_cancer_parent", "Pediatric Cancer - Parent", "Pediatric", 
                      "Parent or guardian of pediatric cancer patient - comprehensive information",
                      attention_markers=ATTENTION_MARKERS["pediatric_cancer_parent"]),
        PatientProfile("pediatric_cancer_teen", "Pediatric Cancer - Teen", "Pediatric", 
                      "Adolescent cancer patient - balancing honesty with hope",
                      is_pediatric=True, attention_markers=ATTENTION_MARKERS["default"]),
        PatientProfile("pediatric_general", "General Pediatric", "Pediatric", 
                      "General child patient - age-appropriate communication",
                      is_pediatric=True, attention_markers=ATTENTION_MARKERS["default"]),
        PatientProfile("pediatric_anxiety", "Pediatric Anxiety", "Pediatric", 
                      "Child with anxiety - extra reassurance needed",
                      is_pediatric=True, attention_markers=ATTENTION_MARKERS["anxiety"]),
        PatientProfile("pediatric_autism", "Pediatric Autism", "Pediatric", 
                      "Autistic child - sensory accommodations, structured information",
                      is_pediatric=True, attention_markers=ATTENTION_MARKERS["autism"]),
        PatientProfile("pediatric_adhd", "Pediatric ADHD", "Pediatric", 
                      "Child with ADHD - engaging, active content",
                      is_pediatric=True, attention_markers=ATTENTION_MARKERS["adhd"]),
    ],
    "Mental Health": [
        PatientProfile("depression", "Major Depression", "Mental Health", 
                      "Patient with major depressive disorder - hope-oriented messaging",
                      attention_markers=ATTENTION_MARKERS["depression"]),
        PatientProfile("anxiety", "Generalized Anxiety Disorder", "Mental Health", 
                      "Patient with chronic anxiety - grounding, manageable language",
                      attention_markers=ATTENTION_MARKERS["anxiety"]),
        PatientProfile("bipolar", "Bipolar Disorder", "Mental Health", 
                      "Patient with bipolar disorder - balanced information",
                      attention_markers=ATTENTION_MARKERS["default"]),
        PatientProfile("ptsd", "PTSD", "Mental Health", 
                      "Patient with post-traumatic stress disorder - trauma-informed",
                      attention_markers=ATTENTION_MARKERS["default"]),
        PatientProfile("eating_disorder", "Eating Disorder", "Mental Health", 
                      "Patient with eating disorder - non-triggering language",
                      attention_markers=ATTENTION_MARKERS["default"]),
    ],
    "Neurodiverse": [
        PatientProfile("autism", "Autism Spectrum", "Neurodiverse", 
                      "Autistic patient - structured, literal, sensory-aware information",
                      attention_markers=ATTENTION_MARKERS["autism"]),
        PatientProfile("adhd", "ADHD", "Neurodiverse", 
                      "Patient with ADHD - engaging, chunked, strength-based content",
                      attention_markers=ATTENTION_MARKERS["adhd"]),
        PatientProfile("dyslexia", "Dyslexia", "Neurodiverse", 
                      "Patient with dyslexia - visual supports, audio alternatives",
                      attention_markers=ATTENTION_MARKERS["dyslexia"]),
        PatientProfile("tourette", "Tourette Syndrome", "Neurodiverse", 
                      "Patient with Tourettes - tic-accommodating, destigmatizing",
                      attention_markers=ATTENTION_MARKERS["default"]),
    ],
    "Chronic Conditions": [
        PatientProfile("diabetes", "Diabetes", "Chronic Conditions", 
                      "Patient managing diabetes - empowerment-focused",
                      attention_markers=ATTENTION_MARKERS["diabetes"]),
        PatientProfile("hypertension", "Hypertension", "Chronic Conditions", 
                      "Patient with high blood pressure - lifestyle emphasis",
                      attention_markers=ATTENTION_MARKERS["default"]),
        PatientProfile("heart_disease", "Heart Disease", "Chronic Conditions", 
                      "Patient with cardiovascular condition - clear action steps",
                      attention_markers=ATTENTION_MARKERS["default"]),
        PatientProfile("copd", "COPD", "Chronic Conditions", 
                      "Patient with chronic obstructive pulmonary disease - breathing focus",
                      attention_markers=ATTENTION_MARKERS["default"]),
        PatientProfile("kidney_disease", "Kidney Disease", "Chronic Conditions", 
                      "Patient with kidney disease - diet and fluid awareness",
                      attention_markers=ATTENTION_MARKERS["default"]),
        PatientProfile("arthritis", "Arthritis", "Chronic Conditions", 
                      "Patient with arthritis - mobility and pain management",
                      attention_markers=ATTENTION_MARKERS["default"]),
    ],
    "General Adult": [
        PatientProfile("adult_standard", "Standard Adult", "General Adult", 
                      "Typical adult patient with average health literacy",
                      attention_markers=ATTENTION_MARKERS["default"]),
        PatientProfile("adult_low_literacy", "Low Health Literacy", "General Adult", 
                      "Adult with limited health literacy - simplified language",
                      attention_markers=ATTENTION_MARKERS["default"]),
        PatientProfile("adult_high_literacy", "High Health Literacy", "General Adult", 
                      "Highly educated patient - detailed, technical information",
                      attention_markers=ATTENTION_MARKERS["default"]),
        PatientProfile("elderly", "Geriatric", "General Adult", 
                      "Senior patient - larger text suggestions, clear instructions",
                      attention_markers=ATTENTION_MARKERS["default"]),
    ],
    "Cancer Care": [
        PatientProfile("adult_cancer", "Adult Cancer Patient", "Cancer Care", 
                      "Adult undergoing cancer treatment - balanced honesty",
                      attention_markers=ATTENTION_MARKERS["default"]),
        PatientProfile("cancer_survivor", "Cancer Survivor", "Cancer Care", 
                      "Post-treatment survivorship care - monitoring and wellness",
                      attention_markers=ATTENTION_MARKERS["default"]),
        PatientProfile("palliative_care", "Palliative Care", "Cancer Care", 
                      "Patient receiving palliative care - comfort and dignity focused",
                      attention_markers=ATTENTION_MARKERS["default"]),
    ],
    "Special Populations": [
        PatientProfile("pregnancy", "Pregnancy", "Special Populations", 
                      "Pregnant patient - prenatal care guidance, safety-focused",
                      attention_markers=ATTENTION_MARKERS["default"]),
        PatientProfile("substance_recovery", "Substance Use Recovery", "Special Populations", 
                      "Patient in recovery - non-judgmental, supportive",
                      attention_markers=ATTENTION_MARKERS["default"]),
        PatientProfile("immunocompromised", "Immunocompromised", "Special Populations", 
                      "Patient with weakened immune system - infection prevention",
                      attention_markers=ATTENTION_MARKERS["default"]),
        PatientProfile("visual_impairment", "Visual Impairment", "Special Populations", 
                      "Patient with vision loss - audio/tactile alternatives",
                      attention_markers=ATTENTION_MARKERS["default"]),
        PatientProfile("hearing_impairment", "Hearing Impairment", "Special Populations", 
                      "Patient with hearing loss - visual communication emphasis",
                      attention_markers=ATTENTION_MARKERS["default"]),
    ],
}

ALL_PROFILES = {}
for cat, profs in PROFILES.items():
    for p in profs:
        ALL_PROFILES[p.id] = p

CATEGORIES = ["All", "Pediatric", "Cancer Care", "Mental Health", "Neurodiverse", "Chronic Conditions", "General Adult", "Special Populations"]

# =============================================================================
# FHIR PARSING & CONTRAINDICATION DETECTION
# =============================================================================
def parse_fhir_bundle(fhir_json: str) -> Dict[str, List[str]]:
    """Parse FHIR Bundle to extract medications, conditions, and allergies."""
    try:
        bundle = json.loads(fhir_json)
        entries = bundle.get("entry", [])
        
        medications = []
        conditions = []
        allergies = []
        
        for entry in entries:
            resource = entry.get("resource", {})
            resource_type = resource.get("resourceType", "")
            
            if resource_type == "MedicationRequest":
                med_code = resource.get("medicationCodeableConcept", {})
                med_name = med_code.get("text", "").lower()
                if not med_name:
                    coding = med_code.get("coding", [{}])[0]
                    med_name = coding.get("display", "").lower()
                if med_name:
                    medications.append(med_name)
                    
            elif resource_type == "Condition":
                cond_code = resource.get("code", {})
                cond_name = cond_code.get("text", "").lower()
                if not cond_name:
                    coding = cond_code.get("coding", [{}])[0]
                    cond_name = coding.get("display", "").lower()
                if cond_name:
                    conditions.append(cond_name)
                    
            elif resource_type == "AllergyIntolerance":
                allergy_code = resource.get("code", {})
                allergy_name = allergy_code.get("text", "").lower()
                if not allergy_name:
                    coding = allergy_code.get("coding", [{}])[0]
                    allergy_name = coding.get("display", "").lower()
                if allergy_name:
                    allergies.append(allergy_name)
        
        return {
            "medications": medications,
            "conditions": conditions,
            "allergies": allergies
        }
    except Exception as e:
        return {"medications": [], "conditions": [], "allergies": [], "error": str(e)}

def detect_contraindications(fhir_data: Dict) -> List[Contraindication]:
    """Detect contraindications from FHIR data against known interactions."""
    contras = []
    medications = fhir_data.get("medications", [])
    conditions = fhir_data.get("conditions", [])
    allergies = fhir_data.get("allergies", [])
    
    # Drug-drug interactions
    for med1, med2, desc in KNOWN_CONTRAINDICATIONS["drug_drug"]:
        if any(med1 in m for m in medications) and any(med2 in m for m in medications):
            contras.append(Contraindication(
                type="drug_drug",
                severity="high",
                description=desc,
                involved_items=[med1, med2]
            ))
    
    # Drug-allergy interactions
    for drug, allergy_list, desc in KNOWN_CONTRAINDICATIONS["drug_allergy"]:
        if any(drug in m for m in medications):
            for allergy in allergies:
                if any(a in allergy for a in allergy_list.split(",")):
                    contras.append(Contraindication(
                        type="drug_allergy",
                        severity="high",
                        description=desc,
                        involved_items=[drug, allergy]
                    ))
    
    # Drug-condition interactions
    for drug, condition, desc in KNOWN_CONTRAINDICATIONS["drug_condition"]:
        if any(drug in m for m in medications):
            if any(condition in c for c in conditions):
                contras.append(Contraindication(
                    type="drug_condition",
                    severity="high" if "kidney" in condition or "asthma" in condition else "medium",
                    description=desc,
                    involved_items=[drug, condition]
                ))
    
    return contras

# =============================================================================
# ATTENTION SHIFT METRICS CALCULATION
# =============================================================================
def calculate_attention_shift(profile: PatientProfile, content: str) -> Dict[str, Any]:
    """Calculate attention shift metrics to prove archetype adaptation."""
    markers = profile.attention_markers if profile.attention_markers else ATTENTION_MARKERS["default"]
    content_lower = content.lower()
    words = content_lower.split()
    word_count = len(words)
    
    # Priority coverage: how many priority keywords appear
    priority_keywords = markers.get("priority_keywords", [])
    priority_found = sum(1 for kw in priority_keywords if kw.lower() in content_lower)
    priority_coverage = priority_found / len(priority_keywords) if priority_keywords else 0
    
    # Avoid keywords check (should be 0)
    avoid_keywords = markers.get("avoid_keywords", [])
    avoid_found = sum(1 for kw in avoid_keywords if kw.lower() in content_lower)
    
    # Trigger density: emotional triggers per 100 words
    emotional_triggers = markers.get("emotional_triggers", [])
    trigger_count = sum(1 for t in emotional_triggers if t.lower() in content_lower)
    trigger_density = (trigger_count / word_count * 100) if word_count > 0 else 0
    
    # Entropy: measure of content adaptation (1 - normalized coverage)
    entropy = 1 - priority_coverage
    
    # Archetype shift detected if priority coverage > 50% and no avoid words
    archetype_shift = priority_coverage > 0.5 and avoid_found == 0
    
    return {
        "entropy": round(entropy, 3),
        "trigger_density": round(trigger_density, 2),
        "priority_coverage": round(priority_coverage * 100, 1),
        "avoid_violations": avoid_found,
        "archetype_shift_detected": archetype_shift,
        "word_count": word_count,
        "priority_keywords_found": priority_found
    }

# =============================================================================
# MULTI-AGENT WORKFLOW
# =============================================================================
def generate_text(prompt: str, temperature: float = 0.4, max_tokens: int = 2000) -> Dict[str, Any]:
    """Generate text using MedGemma with proper GPU memory management."""
    if not HF_API_TOKEN:
        return {"success": False, "error": "HF_TOKEN not set", "content": ""}
    
    try:
        tokenizer, model = load_medgemma(HF_API_TOKEN)
        messages = [{"role": "user", "content": prompt}]
        inputs = tokenizer.apply_chat_template(
            messages, add_generation_prompt=True, tokenize=True, 
            return_tensors="pt", return_dict=True
        ).to(model.device)

        outputs = model.generate(
            **inputs, max_new_tokens=max_tokens, 
            do_sample=True, temperature=max(temperature, 0.01),  # Floor at 0.01
            top_p=0.95, pad_token_id=tokenizer.eos_token_id  # Prevent endless generation
        )

        input_len = inputs['input_ids'].shape[1]
        response = tokenizer.decode(outputs[0][input_len:], skip_special_tokens=True)
        
        return {"success": True, "content": response.strip()}
        
    except Exception as e:
        return {"success": False, "error": str(e), "content": ""}
    finally:
        # CRITICAL: GPU memory cleanup to prevent OOM
        if 'inputs' in locals():
            del inputs
        if 'outputs' in locals():
            del outputs
        torch.cuda.empty_cache()
        gc.collect()

def writer_agent(clinical_summary: str, profile: PatientProfile, config: Dict, 
                 is_parent_version: bool = False) -> Dict[str, Any]:
    """Writer Agent: Generates personalized content (temp=0.4)."""
    
    tone_map = {1: "Formal/Medical", 3: "Professional", 5: "Warm/Supportive", 7: "Friendly", 10: "Casual/Conversational"}
    tone = tone_map.get(min(config['tone'] // 2 * 2 + 1, 10), "Warm/Supportive")
    
    # Archetype-specific instructions
    archetype_instructions = {
        "pediatric_cancer_child": "Use gentle, age-appropriate language. Include story elements and comforting imagery. Avoid statistics and survival rates.",
        "pediatric_cancer_parent": "Provide comprehensive information including prognosis, treatment options, and support resources. Be honest but hopeful.",
        "depression": "Use hope-oriented language. Emphasize that depression is treatable and the patient is not alone. Avoid phrases like 'just cheer up'.",
        "anxiety": "Include grounding techniques and coping strategies. Use reassuring, manageable language.",
        "autism": "Use literal, structured language. Include sensory considerations. Avoid 'cure' language - focus on support and accommodation.",
        "adhd": "Use engaging, chunked content. Highlight strengths and positive traits. Include practical strategies.",
        "dyslexia": "Suggest audio alternatives and visual supports. Emphasize strengths and alternative learning methods.",
    }
    
    archetype_note = archetype_instructions.get(profile.id, "Adapt content for this patient's specific needs.")
    
    version_note = ""
    if is_parent_version:
        version_note = "\n\nThis is the PARENT/GUARDIAN version. Include comprehensive medical information, prognosis, treatment options, and support resources."
    elif profile.is_dual_brochure and not is_parent_version:
        version_note = "\n\nThis is the CHILD version. Use very gentle, age-appropriate language with story elements. Avoid scary medical terms and statistics."
    
    prompt = f"""You are a medical communication specialist. Create a personalized patient education brochure.

PATIENT PROFILE:
- Type: {profile.name}
- Description: {profile.description}

CONFIGURATION:
- Health Literacy Level: {config['literacy']}/10 (1=Basic, 10=Advanced)
- Communication Tone: {tone}
- Empathy Level: {config['empathy']}/10 (1=Clinical, 10=Highly Supportive)
- Detail Level: {config['detail']}/10 (1=Overview, 10=Comprehensive)
- Use Analogies: {'Yes' if config['analogies'] else 'No'}
- Include Visual Descriptions: {'Yes' if config['visual'] else 'No'}
- Include Statistics: {'Yes' if config['statistics'] else 'No'}
- Include Prognosis: {'Yes' if config['prognosis'] else 'No'}

ARCHETYPE GUIDANCE:
{archetype_note}{version_note}

CLINICAL INFORMATION:
{clinical_summary}

Create a well-structured brochure with:
1. A welcoming title
2. "What This Means" section (explanation in plain language)
3. "What to Expect" section (treatment/process overview)
4. "What You Can Do" section (actionable steps)
5. "Questions to Ask Your Doctor" section
6. "Resources & Support" section

Format with clear headings and short paragraphs."""

    return generate_text(prompt, temperature=0.4, max_tokens=2500)

def verifier_agent(clinical_summary: str, brochure_content: str, 
                   contraindications: List[Dict]) -> Dict[str, Any]:
    """Medical Verifier Agent: Safety validation (temp=0.0 for determinism)."""
    
    # Format contraindications for the prompt
    contra_section = ""
    if contraindications:
        contra_section = "\n\nDETECTED CONTRAINDICATIONS:\n"
        for c in contraindications:
            contra_section += f"- [{c.get('severity', 'HIGH').upper()}] {c.get('description')}\n"
    
    prompt = f"""You are a medical safety verifier. Review this patient brochure for accuracy and safety.

ORIGINAL CLINICAL INFORMATION:
{clinical_summary}

BROCHURE TO VERIFY:
{brochure_content[:2000]}
{contra_section}

VERIFICATION CHECKLIST:
1. Are all medical facts accurate based on the clinical information?
2. Is the tone appropriate and non-harmful?
3. Are there any contradictions with the clinical summary?
4. Are contraindications properly addressed (if any)?
5. Is the advice safe and appropriate for the patient's condition?

Respond in this exact format:
VERIFIED: YES or NO
CONFIDENCE: 0-100
SAFETY_ISSUES: List any safety concerns or "None"
INACCURACIES: List any factual errors or "None"
RECOMMENDATIONS: Brief suggestions for improvement or "None"""

    # Call MedGemma with Temperature 0.0 for deterministic results
    result = generate_text(prompt, temperature=0.0, max_tokens=500)
    
    # --- IMPROVED PARSING LOGIC ---
    content = result.get("content", "")
    # Check for "YES" in a way that isn't broken by conversational filler
    verified = "verified: yes" in content.lower()[:100]
    
    # Extract confidence score using regex
    confidence_match = re.search(r'CONFIDENCE:\s*(\d+)', content)
    confidence = int(confidence_match.group(1)) if confidence_match else (90 if verified else 40)
    
    return {
        "verified": verified,
        "confidence": confidence,
        "issues": [content.split("ISSUES:")[1].strip()] if not verified and "ISSUES:" in content else [],
        "raw_response": content
    }

def translator_agent(content: str, target_language: str) -> Dict[str, Any]:
    """Translator Agent: Post-verification translation (temp=0.3)."""
    
    if target_language == "English":
        return {"success": True, "content": content}
    
    prompt = f"""Translate the following medical brochure into {target_language}.

IMPORTANT TRANSLATION GUIDELINES:
- Maintain medical accuracy - use proper medical terminology in {target_language}
- Preserve the tone and empathy level
- Keep the structure and formatting
- Ensure cultural appropriateness
- Do not add or remove information

BROCHURE TO TRANSLATE:
{content}

Provide the complete translated brochure:"""

    return generate_text(prompt, temperature=0.3, max_tokens=3000)

# =============================================================================
# MAIN GENERATION PIPELINE
# =============================================================================
def generate_brochure(clinical_summary: str, profile: PatientProfile, config: Dict, 
                      fhir_data: str = "") -> GeneratedBrochure:
    """Execute full multi-agent pipeline with st.status visibility."""
    
    generation_log = []
    session_id = str(uuid.uuid4())[:8]
    
    # Parse FHIR and detect contraindications
    contraindications = []
    fhir_parsed = {"medications": [], "conditions": [], "allergies": []}
    
    if fhir_data.strip():
        with st.status("Parsing FHIR data and checking contraindications...", expanded=True) as status:
            fhir_parsed = parse_fhir_bundle(fhir_data)
            contraindications = detect_contraindications(fhir_parsed)
            
            if contraindications:
                status.update(label=f"Found {len(contraindications)} contraindication(s)", state="complete")
                for c in contraindications:
                    st.warning(f"[{c.severity.upper()}] {c.description}")
            else:
                status.update(label="No contraindications detected", state="complete")
    
    # Writer Agent
    with st.status("Writer Agent: Generating personalized content...", expanded=True) as status:
        writer_result = writer_agent(clinical_summary, profile, config)
        
        if not writer_result["success"]:
            status.update(label="Writer Agent failed", state="error")
            return GeneratedBrochure(
                patient_content=f"Error: {writer_result['error']}",
                profile_id=profile.id,
                session_id=session_id,
                contraindications=[{"type": c.type, "description": c.description} for c in contraindications]
            )
        
        english_content = writer_result["content"]
        generation_log.append({"agent": "Writer", "status": "success", "tokens": len(english_content.split())})
        status.update(label="Writer Agent complete", state="complete")
    
    # Calculate attention metrics on English content
    attention_metrics = calculate_attention_shift(profile, english_content)
    
    # Dual brochure for pediatric cancer
    parent_content = None
    if profile.is_dual_brochure:
        with st.status("Writer Agent: Generating parent version...", expanded=True) as status:
            parent_result = writer_agent(clinical_summary, profile, config, is_parent_version=True)
            if parent_result["success"]:
                parent_content = parent_result["content"]
                generation_log.append({"agent": "Writer (Parent)", "status": "success"})
            status.update(label="Parent version complete", state="complete")
    
    # Medical Verifier Agent
    with st.status("Medical Verifier: Checking safety and accuracy...", expanded=True) as status:
        verification = verifier_agent(clinical_summary, english_content, contraindications)
        generation_log.append({"agent": "Verifier", "status": "success", "verified": verification["verified"]})
        
        if verification["verified"]:
            status.update(label=f"✅ Verified (Confidence: {verification['confidence']}%)", state="complete", expanded=False)
        else:
    # Use 'error' or 'complete' but keep the warning text in the label
            status.update(label=f"⚠️ Verification Issues (Confidence: {verification['confidence']}%)", state="complete", expanded=True)
            if verification["issues"]:
                for issue in verification["issues"]:
                    st.warning(issue) # Use the warning widget inside the status container instead
    
    # Translator Agent (runs AFTER verification on English content)
    final_content = english_content
    if config['language'] != "English":
        with st.status(f"Translator Agent: Translating to {config['language']}...", expanded=True) as status:
            translator_result = translator_agent(english_content, config['language'])
            
            if translator_result["success"]:
                final_content = translator_result["content"]
                generation_log.append({"agent": "Translator", "status": "success"})
                status.update(label=f"Translation to {config['language']} complete", state="complete")
            else:
                generation_log.append({"agent": "Translator", "status": "failed", "error": translator_result.get("error")})
                status.update(label="Translation failed - using English", state="warning")
    
    # Translate parent content if exists
    final_parent_content = parent_content
    if parent_content and config['language'] != "English":
        with st.status(f"Translator Agent: Translating parent version...", expanded=True) as status:
            parent_translator_result = translator_agent(parent_content, config['language'])
            if parent_translator_result["success"]:
                final_parent_content = parent_translator_result["content"]
            status.update(label="Parent translation complete", state="complete")
    
    return GeneratedBrochure(
        patient_content=final_content,
        parent_content=final_parent_content,
        verification_result=verification,
        attention_metrics=attention_metrics,
        session_id=session_id,
        profile_id=profile.id,
        doctor_approved=False,
        generation_log=generation_log,
        contraindications=[{"type": c.type, "severity": c.severity, "description": c.description} for c in contraindications]
    )

# =============================================================================
# UI COMPONENTS
# =============================================================================
def get_profile_tags(profile: PatientProfile) -> List[Tuple[str, str]]:
    """Generate tags for profile cards based on profile characteristics."""
    tags = []
    
    # Empathy level based on category
    if "Cancer" in profile.name or profile.category in ["Mental Health"]:
        tags.append(("Empathy: 10/10", "tag-pink"))
    elif profile.category in ["Neurodiverse", "Special Populations", "Pediatric"]:
        tags.append(("Empathy: 9/10", "tag-pink"))
    else:
        tags.append(("Empathy: 7/10", "tag-pink"))
    
    # Complexity level
    if "Child" in profile.name or "Low" in profile.name:
        tags.append(("Simplified", "tag-yellow"))
    elif "High" in profile.name or "Parent" in profile.name:
        tags.append(("Advanced", "tag-blue"))
    else:
        tags.append(("Standard", "tag-gray"))
    
    # Special indicators
    if profile.is_dual_brochure:
        tags.append(("Dual Brochure", "tag-green"))
    if profile.is_pediatric:
        tags.append(("Pediatric", "tag-blue"))
    
    return tags

def render_nav():
    """Render top navigation tabs."""
    tab_labels = ["Profiles", "Configuration", "Input Data", "Results", "Architecture"]
    tab_ids = ["profiles", "config", "input", "results", "architecture"]
    
    cols = st.columns(len(tab_labels))
    for i, (label, tab_id) in enumerate(zip(tab_labels, tab_ids)):
        with cols[i]:
            btn_type = "primary" if st.session_state.current_tab == tab_id else "secondary"
            if st.button(label, key=f"nav_{tab_id}", type=btn_type, use_container_width=True):
                st.session_state.current_tab = tab_id
                st.rerun()

def render_profiles():
    """Render profile selection with category pills."""
    # Category Pills
    pill_cols = st.columns(len(CATEGORIES))
    for i, cat in enumerate(CATEGORIES):
        with pill_cols[i]:
            btn_type = "primary" if st.session_state.selected_category == cat else "secondary"
            if st.button(cat, key=f"pill_{cat}", type=btn_type, use_container_width=True):
                st.session_state.selected_category = cat
                st.rerun()
    
    # Profile Cards by Category
    for category, profiles in PROFILES.items():
        if st.session_state.selected_category != "All" and category != st.session_state.selected_category:
            continue
        
        st.markdown(f'<div class="section-title">{category.upper()}</div>', unsafe_allow_html=True)
        
        cols = st.columns(2)
        for idx, profile in enumerate(profiles):
            with cols[idx % 2]:
                is_selected = st.session_state.selected_profile == profile.id
                tags = get_profile_tags(profile)
                
                card_class = "profile-card selected" if is_selected else "profile-card"
                tags_html = "".join([f'<span class="tag {cls}">{txt}</span>' for txt, cls in tags])
                
                st.markdown(f"""
                <div class="{card_class}">
                    <div class="profile-name">{profile.name}</div>
                    <div class="profile-desc">{profile.description}</div>
                    <div class="profile-tags">{tags_html}</div>
                </div>
                """, unsafe_allow_html=True)
                
                if st.button("Select", key=f"sel_{profile.id}", use_container_width=True):
                    st.session_state.selected_profile = profile.id
                    st.rerun()

def render_config():
    """Render configuration sliders and toggles."""
    # Row 1: Health Literacy and Tone
    c1, c2 = st.columns(2)
    with c1:
        with st.container(border=True):
            st.markdown("**Health Literacy**")
            lit = st.slider("Literacy", 1, 10, st.session_state.config.get('literacy', 5), 
                          key="lit_slider", label_visibility="collapsed")
            st.session_state.config['literacy'] = lit
            st.markdown(f'<div class="slider-labels"><span>Basic</span><span class="slider-value">{lit}/10</span><span>Advanced</span></div>', unsafe_allow_html=True)
    
    with c2:
        with st.container(border=True):
            st.markdown("**Tone**")
            tone = st.slider("Tone", 1, 10, st.session_state.config.get('tone', 5), 
                           key="tone_slider", label_visibility="collapsed")
            st.session_state.config['tone'] = tone
            st.markdown(f'<div class="slider-labels"><span>Formal</span><span class="slider-value">{tone}/10</span><span>Conversational</span></div>', unsafe_allow_html=True)
    
    # Row 2: Empathy and Detail
    c3, c4 = st.columns(2)
    with c3:
        with st.container(border=True):
            st.markdown("**Empathy Level**")
            emp = st.slider("Empathy", 1, 10, st.session_state.config.get('empathy', 7), 
                          key="emp_slider", label_visibility="collapsed")
            st.session_state.config['empathy'] = emp
            st.markdown(f'<div class="slider-labels"><span>Clinical</span><span class="slider-value">{emp}/10</span><span>Supportive</span></div>', unsafe_allow_html=True)
    
    with c4:
        with st.container(border=True):
            st.markdown("**Detail Level**")
            det = st.slider("Detail", 1, 10, st.session_state.config.get('detail', 6), 
                          key="det_slider", label_visibility="collapsed")
            st.session_state.config['detail'] = det
            st.markdown(f'<div class="slider-labels"><span>Overview</span><span class="slider-value">{det}/10</span><span>Comprehensive</span></div>', unsafe_allow_html=True)
    
    # Language Selection
    with st.container(border=True):
        st.markdown("**Output Language**")
        langs = ["English", "Spanish", "French", "German", "Chinese", "Japanese", "Arabic", "Hindi", "Portuguese", "Russian", "Italian", "Korean", "Vietnamese", "Polish", "Dutch", "Turkish", "Thai", "Swedish"]
        lang = st.selectbox("Language", langs, index=langs.index(st.session_state.config.get('language', 'English')), 
                          key="lang_select", label_visibility="collapsed")
        st.session_state.config['language'] = lang
    
    # Toggle Options
    st.markdown("<br>", unsafe_allow_html=True)
    t1, t2 = st.columns(2)
    with t1:
        with st.container(border=True):
            c1a, c1b = st.columns([3, 1])
            with c1a:
                st.markdown("**Use Analogies**")
                st.caption("Explain with relatable comparisons")
            with c1b:
                st.session_state.config['analogies'] = st.toggle("Analogies", 
                    value=st.session_state.config.get('analogies', True), key="tog_analogies", label_visibility="collapsed")
        
        with st.container(border=True):
            c2a, c2b = st.columns([3, 1])
            with c2a:
                st.markdown("**Include Statistics**")
                st.caption("Add relevant data and numbers")
            with c2b:
                st.session_state.config['statistics'] = st.toggle("Statistics", 
                    value=st.session_state.config.get('statistics', False), key="tog_stats", label_visibility="collapsed")
    
    with t2:
        with st.container(border=True):
            c3a, c3b = st.columns([3, 1])
            with c3a:
                st.markdown("**Visual Descriptions**")
                st.caption("Include visual aid suggestions")
            with c3b:
                st.session_state.config['visual'] = st.toggle("Visuals", 
                    value=st.session_state.config.get('visual', True), key="tog_visual", label_visibility="collapsed")
        
        with st.container(border=True):
            c4a, c4b = st.columns([3, 1])
            with c4a:
                st.markdown("**Include Prognosis**")
                st.caption("Add outcome information")
            with c4b:
                st.session_state.config['prognosis'] = st.toggle("Prognosis", 
                    value=st.session_state.config.get('prognosis', True), key="tog_prog", label_visibility="collapsed")

def render_input():
    """Render clinical input section with FHIR support."""
    # Sub-tabs for input type
    input_tab = st.radio("Input Type", ["Clinical Notes", "FHIR Bundle"], 
                        horizontal=True, label_visibility="collapsed")
    
    if input_tab == "Clinical Notes":
        with st.container(border=True):
            st.markdown("**Clinical Notes**")
            notes = st.text_area("Enter clinical notes", 
                               value=st.session_state.get('clinical_notes', ''), 
                               placeholder="Enter clinical notes or doctor's shorthand...", 
                               height=200, key="notes_input", label_visibility="collapsed")
            st.session_state.clinical_notes = notes
            st.caption("Enter patient information, diagnosis, treatment plan, and any other relevant clinical details.")
    else:
        with st.container(border=True):
            st.markdown("**FHIR Bundle (JSON)**")
            fhir = st.text_area("Paste FHIR JSON", 
                              value=st.session_state.get('fhir_data', ''), 
                              placeholder='{"resourceType": "Bundle", "entry": [...]}', 
                              height=200, key="fhir_input", label_visibility="collapsed")
            st.session_state.fhir_data = fhir
            st.caption("Paste a FHIR R4 Bundle JSON to automatically extract medications, conditions, and allergies for contraindication checking.")
    
    # Generate Button
    st.markdown("<br>", unsafe_allow_html=True)
    if st.button("Generate Personalized Brochure", type="primary", use_container_width=True):
        if not st.session_state.get('selected_profile'):
            st.error("Please select a patient profile first.")
        elif not st.session_state.clinical_notes.strip():
            st.error("Please enter clinical notes.")
        else:
            profile = ALL_PROFILES.get(st.session_state.selected_profile)
            brochure = generate_brochure(
                st.session_state.clinical_notes, 
                profile, 
                st.session_state.config,
                st.session_state.get('fhir_data', '')
            )
            st.session_state.current_brochure = brochure
            st.session_state.current_tab = "results"
            st.rerun()

def render_results():
    """Render generated brochure with verification status and metrics."""
    if 'current_brochure' not in st.session_state:
        st.info("No brochure generated yet. Go to Input Data to generate one.")
        return
    
    b = st.session_state.current_brochure
    v = b.verification_result
    
    # Status Card
    with st.container(border=True):
        c1, c2 = st.columns([3, 1])
        with c1:
            st.markdown("**Verification Results**")
        with c2:
            if b.doctor_approved:
                st.markdown('<span class="status-approved">Doctor Approved</span>', unsafe_allow_html=True)
            elif v.get('verified'):
                st.markdown('<span class="status-pending">Pending Approval</span>', unsafe_allow_html=True)
            else:
                st.markdown('<span class="status-failed">Verification Failed</span>', unsafe_allow_html=True)
        
        # Metrics
        st.markdown("<br>", unsafe_allow_html=True)
        m1, m2, m3, m4 = st.columns(4)
        with m1:
            st.markdown(f'<div class="metric"><div class="metric-val">{"Yes" if v.get("verified") else "No"}</div><div class="metric-lbl">Verified</div></div>', unsafe_allow_html=True)
        with m2:
            st.markdown(f'<div class="metric"><div class="metric-val">{v.get("confidence", 0)}%</div><div class="metric-lbl">Confidence</div></div>', unsafe_allow_html=True)
        with m3:
            contra_count = len(b.contraindications) if b.contraindications else 0
            st.markdown(f'<div class="metric"><div class="metric-val">{contra_count}</div><div class="metric-lbl">Contraindications</div></div>', unsafe_allow_html=True)
        with m4:
            shift = "Yes" if b.attention_metrics.get("archetype_shift_detected") else "No"
            st.markdown(f'<div class="metric"><div class="metric-val">{shift}</div><div class="metric-lbl">Archetype Adapted</div></div>', unsafe_allow_html=True)
    
    # Contraindication Alerts
    if b.contraindications:
        for contra in b.contraindications:
            severity_color = "contra-alert" if contra.get('severity') == 'high' else "warning"
            st.markdown(f"""
            <div class="contra-alert">
                <div class="contra-title">[{contra.get('severity', 'MEDIUM').upper()}] {contra.get('type', '').replace('_', ' ').title()}</div>
                <div class="contra-list">{contra.get('description', '')}</div>
            </div>
            """, unsafe_allow_html=True)
    
    # Attention Metrics
    with st.expander("Attention Shift Metrics"):
        m = b.attention_metrics
        st.markdown(f"""
        **Entropy:** {m.get('entropy', 'N/A')} (content adaptation measure)
        
        **Trigger Density:** {m.get('trigger_density', 'N/A')}% (emotional triggers per 100 words)
        
        **Priority Coverage:** {m.get('priority_coverage', 'N/A')}% (archetype keyword match)
        
        **Avoid Violations:** {m.get('avoid_violations', 'N/A')} (inappropriate words found)
        
        **Word Count:** {m.get('word_count', 'N/A')}
        
        **Priority Keywords Found:** {m.get('priority_keywords_found', 'N/A')}
        """)
    
    # Generation Log
    with st.expander("Generation Workflow Log"):
        for log in b.generation_log:
            status_icon = "[OK]" if log.get('status') == 'success' else "[FAIL]"
            st.text(f"{status_icon} {log.get('agent')}")
    
    # Patient Content
    with st.container(border=True):
        st.markdown("**Generated Brochure**")
        st.markdown(f'<div class="brochure-content">{b.patient_content}</div>', unsafe_allow_html=True)
    
    # Parent Content (if dual brochure)
    if b.parent_content:
        with st.container(border=True):
            st.markdown("**Parent/Guardian Version**")
            st.markdown(f'<div class="brochure-content">{b.parent_content}</div>', unsafe_allow_html=True)
    
    # Doctor Approval
    st.markdown("<br>", unsafe_allow_html=True)
    col1, col2 = st.columns(2)
    with col1:
        if st.button("Approve (Doctor)", type="primary", use_container_width=True):
            st.session_state.current_brochure.doctor_approved = True
            st.success("Brochure approved by doctor.")
            st.rerun()
    with col2:
        if st.button("Reject (Doctor)", type="secondary", use_container_width=True):
            st.session_state.current_brochure.verification_result["verified"] = False
            st.error("Brochure rejected. Please regenerate with different parameters.")
            st.rerun()
    
    # Export
    st.markdown("<br>", unsafe_allow_html=True)
    export_data = {
        "content": b.patient_content,
        "parent_content": b.parent_content,
        "verification": v,
        "attention_metrics": b.attention_metrics,
        "contraindications": b.contraindications,
        "generation_log": b.generation_log,
        "session_id": b.session_id,
        "profile_id": b.profile_id,
        "doctor_approved": b.doctor_approved,
        "timestamp": datetime.now().isoformat()
    }
    st.download_button("Download JSON", json.dumps(export_data, indent=2), 
                      f"brochure_{b.session_id}.json", use_container_width=True)

def render_architecture():
    """Render architecture explanation for judges."""
    st.markdown("## Multi-Agent Architecture")
    st.markdown("Med0wl uses a 3-agent pipeline with doctor-in-the-loop approval.")
    
    st.markdown("""
    <div class="arch-step">
        <div class="arch-step-title">1. Clinical Input & FHIR Parser</div>
        <div class="arch-step-desc">Accepts clinical notes or FHIR Bundle JSON. Extracts medications, conditions, allergies for contraindication detection.</div>
    </div>
    <div class="arch-arrow">↓</div>
    <div class="arch-step">
        <div class="arch-step-title">2. Writer Agent (Temperature = 0.4)</div>
        <div class="arch-step-desc">Generates personalized content based on 28 patient archetypes. Uses attention markers for archetype-specific adaptation. Supports dual-brochure mode for pediatric cancer.</div>
    </div>
    <div class="arch-arrow">↓</div>
    <div class="arch-step">
        <div class="arch-step-title">3. Medical Verifier (Temperature = 0.0)</div>
        <div class="arch-step-desc">Deterministic safety validation. Checks accuracy against clinical input, detects contraindications, validates tone appropriateness.</div>
    </div>
    <div class="arch-arrow">↓</div>
    <div class="arch-step">
        <div class="arch-step-title">4. Translator Agent (Temperature = 0.3)</div>
        <div class="arch-step-desc">Post-verification translation to 18 languages. Runs ONLY on verified English content to maintain safety.</div>
    </div>
    <div class="arch-arrow">↓</div>
    <div class="arch-step">
        <div class="arch-step-title">5. Doctor-in-the-Loop Approval</div>
        <div class="arch-step-desc">Final human review checkpoint. Doctors can approve or reject before patient delivery.</div>
    </div>
    <div class="arch-arrow">↓</div>
    <div class="arch-step">
        <div class="arch-step-title">6. Output & Audit Trail</div>
        <div class="arch-step-desc">Final brochure with complete generation log, attention metrics, and verification results. JSON export for EHR integration.</div>
    </div>
    """, unsafe_allow_html=True)
    
    st.markdown("### Key Technical Features")
    
    col1, col2 = st.columns(2)
    with col1:
        with st.container(border=True):
            st.markdown("**Attention Shift Metrics**")
            st.markdown("""
            - **Entropy**: Measures content adaptation (1 - priority coverage)
            - **Trigger Density**: Emotional triggers per 100 words
            - **Archetype Shift Detected**: Boolean indicating successful adaptation
            - Proves archetypes aren't just prompt templates
            """)
    with col2:
        with st.container(border=True):
            st.markdown("**Safety Features**")
            st.markdown("""
            - Temperature 0.0 for Medical Verifier (deterministic)
            - FHIR contraindication detection (drug-drug, drug-allergy, drug-condition)
            - GPU memory cleanup after each generation
            - Complete audit trail for regulatory compliance
            """)
    
    st.markdown("### Model Configuration")
    st.code("""
Model: google/medgemma-1.5-4b-it
Device: CUDA (Kaggle T4 GPU)
Dtype: bfloat16
Device Map: {"": 0}  # Single GPU

Agent Temperatures:
- Writer: 0.4 (creative but controlled)
- Verifier: 0.0 (deterministic for safety)
- Translator: 0.3 (consistent translation)
    """)

# =============================================================================
# MAIN
# =============================================================================
def main():
    # Initialize session state
    if 'current_tab' not in st.session_state:
        st.session_state.current_tab = "profiles"
    if 'selected_profile' not in st.session_state:
        st.session_state.selected_profile = None
    if 'selected_category' not in st.session_state:
        st.session_state.selected_category = "All"
    if 'config' not in st.session_state:
        st.session_state.config = {
            'literacy': 5, 'tone': 5, 'empathy': 7, 'detail': 6,
            'language': 'English', 'analogies': True, 'visual': True,
            'statistics': False, 'prognosis': True
        }
    if 'clinical_notes' not in st.session_state:
        st.session_state.clinical_notes = ""
    if 'fhir_data' not in st.session_state:
        st.session_state.fhir_data = ""
    
    # Check HF token
    if not HF_API_TOKEN:
        st.error("Please set HF_TOKEN in Kaggle secrets or environment variables")
        st.stop()
    
    # Header
    st.markdown("# Med0wl - Agentic Patient Communication Engine")
    st.caption("For Kaggle MedGemma Impact Challenge | Authors: Bheant Powar & Ayodeji Ogundele")
    
    # Navigation
    render_nav()
    
    # Content
    if st.session_state.current_tab == "profiles":
        render_profiles()
    elif st.session_state.current_tab == "config":
        render_config()
    elif st.session_state.current_tab == "input":
        render_input()
    elif st.session_state.current_tab == "results":
        render_results()
    elif st.session_state.current_tab == "architecture":
        render_architecture()

if __name__ == "__main__":
    main()


Writing app_final.py


In [3]:
"""from transformers import AutoTokenizer, AutoModelForCausalLM
import torch
from kaggle_secrets import UserSecretsClient

# Get your token specifically for this test
user_secrets = UserSecretsClient()
hf_token = user_secrets.get_secret("HF_TOKEN")

model_id = "google/medgemma-1.5-4b-it"
print("Checking permissions and pre-loading MedGemma...")

try:
    tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)
    model = AutoModelForCausalLM.from_pretrained(
        model_id,
        device_map={"": 0}, 
        torch_dtype=torch.bfloat16,
        token=hf_token  # Explicitly pass the token here
    )
    print("SUCCESS: Model is now in VRAM.")
except Exception as e:
    print(f"FAILED: {e}")"""

'from transformers import AutoTokenizer, AutoModelForCausalLM\nimport torch\nfrom kaggle_secrets import UserSecretsClient\n\n# Get your token specifically for this test\nuser_secrets = UserSecretsClient()\nhf_token = user_secrets.get_secret("HF_TOKEN")\n\nmodel_id = "google/medgemma-1.5-4b-it"\nprint("Checking permissions and pre-loading MedGemma...")\n\ntry:\n    tokenizer = AutoTokenizer.from_pretrained(model_id, token=hf_token)\n    model = AutoModelForCausalLM.from_pretrained(\n        model_id,\n        device_map={"": 0}, \n        torch_dtype=torch.bfloat16,\n        token=hf_token  # Explicitly pass the token here\n    )\n    print("SUCCESS: Model is now in VRAM.")\nexcept Exception as e:\n    print(f"FAILED: {e}")'

In [4]:
# !pip install pyngrok  # Ensure this is installed in your setup cell
import torch
from kaggle_secrets import UserSecretsClient
from pyngrok import ngrok
import subprocess
import time

# 1. Set your Authtoken (Get this from https://dashboard.ngrok.com/get-started/your-authtoken)
user_secrets = UserSecretsClient()
NGROK_AUTH_TOKEN = user_secrets.get_secret("NGROK_TOKEN")
ngrok.set_auth_token(NGROK_AUTH_TOKEN)

# 2. Kill any existing tunnels to prevent port conflicts
ngrok.kill()

# 3. Start Streamlit in the background
# Points to app_final.py created by your %%writefile cell
process = subprocess.Popen([
    "streamlit", "run", "app_final.py", 
    "--server.port=8501", 
    "--server.address=0.0.0.0", 
    "--server.headless=true", 
    "--server.enableCORS=false", 
    "--server.enableXsrfProtection=false",
    "--server.enableWebsocketCompression=false"
])

# 4. Give the server a moment to initialize
time.sleep(5) 

# 5. Open the Ngrok Tunnel
public_url = ngrok.connect(8501).public_url
print(f"============================================================")
print(f"MED0WL IS LIVE AT: {public_url}")
print(f"============================================================")

BackendError: Unexpected response from the service. Response: {'errors': ['No user secrets exist for kernel id 110545704 and label NGROK_TOKEN.'], 'error': {'code': 5}, 'wasSuccessful': False}.